<a href="https://colab.research.google.com/github/kamilsielski6/spark/blob/main/Spark_DataFrame.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Używanie Apache Spark w Google Colab

Pierwszym krokiem jest zainstalowanie biblioteki `pyspark`, która jest interfejsem Pythona do Apache Spark.

In [1]:
pip install pyspark

Po zainstalowaniu `pyspark` możesz utworzyć `SparkSession`, która jest punktem wejścia do programowania Spark z użyciem Dataset i DataFrame API.

In [5]:
from pyspark.sql import SparkSession

# Utwórz SparkSession
spark = SparkSession.builder \
    .appName("ColabSpark") \
    .getOrCreate()

print("SparkSession stworzona pomyślnie!")

SparkSession stworzona pomyślnie!


Teraz możesz wykonywać podstawowe operacje Spark. Na przykład, utwórz prosty DataFrame i wyświetl jego zawartość.

In [3]:
data = [("Jan", 25), ("Anna", 30), ("Piotr", 35)]
columns = ["Imię", "Wiek"]

df = spark.createDataFrame(data, columns)
df.show()

# Zakończ SparkSession, gdy nie jest już potrzebna
# spark.stop()

+-----+----+
| Imię|Wiek|
+-----+----+
|  Jan|  25|
| Anna|  30|
|Piotr|  35|
+-----+----+



In [4]:
spark.stop()

In [6]:
df = spark.read.json('people.json')

In [7]:
df.show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



In [8]:
df.printSchema() # schemat

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)



In [9]:
df.columns # nazwy kolumn

['age', 'name']

In [12]:
df.describe().show() # statystki

+-------+------------------+-------+
|summary|               age|   name|
+-------+------------------+-------+
|  count|                 2|      3|
|   mean|              24.5|   NULL|
| stddev|7.7781745930520225|   NULL|
|    min|                19|   Andy|
|    max|                30|Michael|
+-------+------------------+-------+



## 2. Jawne definiowanie schematu danych

In [13]:
from pyspark.sql.types import StructField, StringType, IntegerType, StructType

In [14]:
data_schema = [StructField('age', IntegerType(), True),
               StructField('name', StringType(), True)]

In [16]:
final_struct = StructType(fields=data_schema)

In [19]:
df = spark.read.json('people.json', schema=final_struct)

In [20]:
df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- name: string (nullable = true)



In [21]:
type(df['age'])

pyspark.sql.classic.column.Column

In [22]:
df.select('age').show()

+----+
| age|
+----+
|NULL|
|  30|
|  19|
+----+



In [23]:
type(df.select('age'))

pyspark.sql.classic.dataframe.DataFrame

In [24]:
df.head(2)

[Row(age=None, name='Michael'), Row(age=30, name='Andy')]

In [26]:
df.select(['age', 'name']).show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



In [28]:
df.withColumn('double_age', df['age']*2).show() # stworzenie nowej kolumny, w tej formie nie nadpisuje pierwotnego dataframe

+----+-------+----------+
| age|   name|double_age|
+----+-------+----------+
|NULL|Michael|      NULL|
|  30|   Andy|        60|
|  19| Justin|        38|
+----+-------+----------+



In [29]:
df.withColumnRenamed('age', 'my_new_age').show()

+----------+-------+
|my_new_age|   name|
+----------+-------+
|      NULL|Michael|
|        30|   Andy|
|        19| Justin|
+----------+-------+



## BESPOŚREDNIE KORZYSTANIE Z SQL W SPARK

In [30]:
df.createOrReplaceTempView('people') # tworzę widok tymczasowy

In [31]:
results = spark.sql("SELECT * FROM people")

In [32]:
results.show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



In [33]:
new_results = spark.sql("SELECT * FROM people WHERE age=30")

In [34]:
new_results.show()

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+

